# Simple Workflow

Below, we provide experiments with the dummy (synthetic data) and the molecular data (`musk`).

## 1. Examples with Dummy Data
### 1.1 Load the dummy data

In [ ]:
from sawmil.data import generate_dummy_bags
import numpy as np
rng = np.random.default_rng(0)

ds = generate_dummy_bags(
    n_pos=100, n_neg=100, inst_per_bag=(5, 15), d=2,
    pos_centers=((+2,+1), (+4,+3)),
    neg_centers=((-1.5,-1.0), (-3.0,+0.5)),
    pos_scales=((2.0, 0.6), (1.2, 0.8)),
    neg_scales=((1.5, 0.5), (2.5, 0.9)),
    pos_intra_rate=(0.25, 0.85),
    ensure_pos_in_every_pos_bag=True,
    neg_pos_noise_rate=(0.00, 0.05),
    pos_neg_noise_rate=(0.00, 0.20),
    outlier_rate=0.1,
    outlier_scale=8.0,
    random_state=42,
)

# Quick sanity:
X_pos, pos_idx = ds.positive_instances()
X_neg, neg_idx = ds.negative_instances()
print("Number of bags:", len(ds.bags))


## 1.2. Fit the model

In [ ]:
from sawmil.kernels import get_kernel, Linear
k = get_kernel("linear") # base (single-instance kernel)
k1 = Linear() # equivalent k == k1
# if you want to use kernels for bags outside of the models
from sawmil.bag_kernels import make_bag_kernel
bag_k = make_bag_kernel(k, normalizer="none", p=1.0) # bag kernel
# Otherwise, it is handled inside each model


#### 1.2.1 Fit NSK with the Linear Kernel

In [ ]:
from sawmil import NSK

k = get_kernel("linear", normalizer="average")
clf = NSK(C=1, kernel=k, 
          # bag kernel settings
          normalizer='average',
          p=1.0,
          # solver settings
          scale_C=True, 
          tol=1e-8, 
          verbose=False, 
          solver='osqp').fit(ds, None)
print("Train acc:", clf.score(ds, ds.y))
# clf.predict(ds), clf.decision_function(ds)

#### 1.2.2 Fit NSK with the RBF Kernel

In [ ]:
k = get_kernel("rbf", gamma=0.8)
clf = NSK(C=10, kernel=k, scale_C=True, tol=1e-8, verbose=False, solver='osqp').fit(ds, None)
print("Train acc:", clf.score(ds, ds.y))

#### 1.2.3 Fit NSK with Combined Kernels

In [ ]:
from sawmil.kernels import Product, Polynomial, Linear, RBF, Sum, Scale

k = Sum(Linear(), 
        Scale(0.5, 
              Product(Polynomial(degree=2), RBF(gamma=1.0))))
clf = NSK(C=100, kernel=k, 
          # params to create a bag kernel
          normalizer="none",
          # svm params
          scale_C=True, 
          tol=1e-8, verbose=False, solver='gurobi').fit(ds, None)
print("Train acc:", clf.score(ds, ds.y))

#### 1.2.3 Fit sMIL with the Linear Kernel

In [ ]:
from sawmil import sMIL
from sawmil.kernels import Linear

In [ ]:
k  = Linear()
clf = sMIL(C=10, kernel=k, 
           # params to create a bag kernel
           normalizer="none",
           # svm params
           scale_C=True, 
           tol=1e-8, verbose=False, solver='osqp').fit(ds, None)

In [ ]:
y = np.array([b.y for b in ds.bags])
# yhat = clf.predict(ds)
print("Train acc:", clf.score(ds, y))

#### 1.2.4. Fit sAwMIL with the Linear kernel

In [ ]:
from sawmil import sAwMIL
from sawmil.kernels import get_kernel

In [ ]:
k = get_kernel('linear')
clf = sAwMIL(C=0.1, kernel=k,
             solver="gurobi", eta=0.95) # here eta is high, since all items in the bag are relevant
clf.fit(ds)
print("Train acc:", clf.score(ds, ds.y))
clf.predict(ds)

## 2. Experiments with the molecular data
### 2.1 Load the `musk` data

In [ ]:
from sawmil.data import load_musk_bags
train_ds, test_ds, scaler = load_musk_bags(standardize=True, test_size=0.3, random_state=42)

### 2.2 Fit models
#### 2.2.1 Fit NSK with Combined Kernel

In [ ]:
from sawmil import NSK
from sawmil.kernels import Linear, Sum, RBF
from sklearn.metrics import matthews_corrcoef as mcc


k = Sum(Linear(),RBF(gamma=0.5))

clf = NSK(C=10, kernel=k, 
          normalizer="none",
          scale_C=True, tol=1e-8, verbose=False, solver='osqp').fit(train_ds, None)

y = ds.y
yhat = clf.predict(test_ds)
print('Matthews Correlation Coefficient:', mcc(y, yhat))

#### 2.2.2 Fit sMIL with Combined Kernel

In [ ]:
from sawmil.kernels import Linear
from sawmil import sMIL

k = Sum(Linear(),RBF(gamma=0.5))
solver_params = {
    #'env': {'LogFile': 'gurobi.log'},
    'model': {'Method': 2, 'Threads': 10},
    #'start': np.zeros(train_ds.n_instances) # Specify a warm start (weights for each instance)
}


clf = sMIL(C=10, kernel=k, scale_C=True, tol=1e-8, verbose=True, solver='gurobi', solver_params=solver_params).fit(train_ds, None)

In [ ]:
y = ds.y
yhat = clf.predict(test_ds)
print('Matthews Correlation Coefficient:', mcc(y, yhat))